# Exploratory Data Analysis (EDA) - NYC Airbnb Dataset

In this notebook, we explore the NYC Airbnb dataset to understand its structure, identify data quality issues, and inform our data cleaning decisions. The goal is to:

1. Load the dataset and inspect its structure
2. Generate a profiling report for an overview of all features
3. Identify outliers, missing values, and data anomalies
4. Decide on appropriate cleaning boundaries (e.g., price range, geographic bounds)

## 1. Import Libraries and Load Data

We start by importing the necessary libraries and loading the raw dataset from Weights & Biases (W&B).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import wandb

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Dataset Overview

Let's look at basic statistics and data types to understand what we're working with.

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Dataset Profiling

We now generate an automated profiling report using `ydata-profiling`. This gives us a comprehensive overview of the dataset including:
- Distribution of each variable
- Missing value analysis
- Correlation between features
- Potential outliers and anomalies

This is an efficient way to get a holistic picture before diving into specific issues.

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="NYC Airbnb Dataset Profiling Report", explorative=True)
profile.to_widgets()

## 4. Key Observations from Profiling

After reviewing the profiling report, here are the key findings:

- **Price outliers**: The `price` column has extreme values. Some listings have very low prices (e.g., $0) which are likely errors or free listings, and some have extremely high prices (e.g., $10,000+) which are unrealistic for short-term rentals.
- **Missing values in `last_review`**: The `last_review` and `reviews_per_month` columns have missing values. These correspond to listings that have never been reviewed. This is expected and not necessarily a data quality issue.
- **Neighbourhood groups**: The dataset covers 5 NYC boroughs (Manhattan, Brooklyn, Queens, Bronx, Staten Island). Manhattan and Brooklyn dominate the listings.
- **Geographic distribution**: Most coordinates fall within expected NYC boundaries, but there may be some outliers that fall outside the NYC area.
- **Room types**: Three categories exist - Entire home/apt, Private room, and Shared room.

## 5. Price Distribution Analysis

Let's examine the price distribution more closely to determine appropriate boundaries for filtering outliers.

In [ ]:
# Price distribution
print(f"Price statistics:")
print(f"  Min: ${df['price'].min()}")
print(f"  Max: ${df['price'].max()}")
print(f"  Mean: ${df['price'].mean():.2f}")
print(f"  Median: ${df['price'].median()}")
print(f"  Std: ${df['price'].std():.2f}")
print(f"\nPercentiles:")
print(df['price'].quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['price'], bins=50, edgecolor='black')
axes[0].set_title('Price Distribution (Full Range)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

# Zoomed in view
axes[1].hist(df[df['price'] <= 500]['price'], bins=50, edgecolor='black')
axes[1].set_title('Price Distribution (≤ $500)')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 6. Cleaning Decision: Price Boundaries

Based on the analysis above, we choose the following price boundaries for filtering:

- **Minimum price: $10** — Listings below $10/night are likely errors, test listings, or not genuine short-term rentals. A price of $0 is clearly invalid.
- **Maximum price: $350** — While there are legitimate luxury listings above this price, the vast majority of NYC Airbnb rentals fall well below $350. Prices above this threshold are extreme outliers that could skew our model. The 95th percentile is typically around $300-350, making this a reasonable upper bound.

These boundaries allow us to focus on the realistic range of short-term rental prices while removing noise from the data.

In [ ]:
min_price = 10
max_price = 350

print(f"Rows before price filtering: {len(df)}")
df_clean = df[(df['price'] >= min_price) & (df['price'] <= max_price)].copy()
print(f"Rows after price filtering: {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

## 7. Geographic Analysis

Let's check the geographic distribution of listings to identify any points that fall outside of NYC boundaries.

In [ ]:
print(f"Longitude range: [{df['longitude'].min()}, {df['longitude'].max()}]")
print(f"Latitude range: [{df['latitude'].min()}, {df['latitude'].max()}]")
print(f"\nExpected NYC boundaries:")
print(f"  Longitude: [-74.25, -73.50]")
print(f"  Latitude: [40.5, 41.2]")

# Check for outliers
geo_outliers = df[
    ~(df['longitude'].between(-74.25, -73.50) & df['latitude'].between(40.5, 41.2))
]
print(f"\nListings outside NYC boundaries: {len(geo_outliers)}")
if len(geo_outliers) > 0:
    print(geo_outliers[['name', 'neighbourhood_group', 'latitude', 'longitude']])

In [ ]:
# Plot geographic distribution
fig, ax = plt.subplots(figsize=(10, 10))
scatter = ax.scatter(df['longitude'], df['latitude'], c=df['price'], 
                     cmap='viridis', alpha=0.3, s=1, vmax=500)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('NYC Airbnb Listings - Geographic Distribution')
plt.colorbar(scatter, label='Price ($)')
plt.show()

## 8. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

## 9. Neighbourhood Group Distribution

In [ ]:
neigh_counts = df['neighbourhood_group'].value_counts()
print(neigh_counts)

fig, ax = plt.subplots(figsize=(8, 5))
neigh_counts.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('Listings by Neighbourhood Group')
ax.set_xlabel('Neighbourhood Group')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Summary and Next Steps

### Key Findings:
1. **Price filtering**: We will filter prices to the range [$10, $350] to remove outliers
2. **Geographic filtering**: Some data points may fall outside NYC boundaries — these need to be removed in the cleaning step
3. **Date conversion**: `last_review` should be converted to datetime format
4. **Missing values**: `last_review` and `reviews_per_month` have missing values (listings with no reviews), which is expected

### Cleaning Steps to Implement:
- Remove listings with price outside [10, 350]
- Convert `last_review` to datetime
- Remove listings outside NYC geographic boundaries (longitude: [-74.25, -73.50], latitude: [40.5, 41.2])

In [ ]:
run.finish()